## For converting CAN messages into human readable information

The DBC files are retrieved from https://github.com/commaai/opendbc/blob/master/opendbc/dbc/

Reference code from https://github.com/seer-lab/xivt-ml-cav

In [ ]:
import pandas as pd

from collections import Counter

import cantools
import re
import can
from can import Message


In [10]:
def openCSV(filename):
    '''
    Takes in filename string (with position relative to this location)
    Returns dataframe
    '''
    #still skips first line
    file = pd.read_csv(filename)
    return pd.DataFrame(file)

In [20]:

# add a clean version of translate that supports CSV exporting

def translate_with_csv(the_df, dbc_name, N=10000, output_csv=None):
    """Decode CAN messages using a DBC and optionally save the decoded/encoded pairs.

    Parameters
    ----------
    the_df : pandas.DataFrame
        CAN dataset formatted.
    dbc_name : str
        Path to the DBC file to use for decoding.
    N : int, optional
        Number of rows from ``the_df`` to process (default 10000).
    output_csv : str or None, optional
        Filename where results will be written as a CSV. If ``None`` no file is written.

    Returns
    -------
    pandas.DataFrame or None
        DataFrame of results if ``output_csv`` is provided, otherwise ``None``.
    """
    total_count = N
    translated_count = 0
    non_translated_count = 0
    results = []

    # load the DBC file
    try:
        dbc = cantools.database.load_file(dbc_name)
    except Exception as e:
        print(f"failed to load DBC '{dbc_name}': {e}")
        # re-raise so caller is aware if they need to fix the file
        raise

    for idx in range(N):
        idDec = int(the_df.iloc[idx, 1], 16)
        payload_str = str(the_df.iloc[idx, 3])
        payload_list = [int(payload_str[i : i + 2], 16) for i in range(0, len(payload_str), 2)]
        label = the_df.iloc[idx]["Label"] if "Label" in the_df.columns else None

        current_message = Message(
            timestamp=the_df.iloc[idx, 0], arbitration_id=idDec, data=payload_list
        )

        try:
            translated_message = dbc.decode_message(
                current_message.arbitration_id, current_message.data
            )
            
            _data = dbc.encode_message(idDec, data=translated_message)
            translated_count += 1

            if output_csv:
                row = {
                    "timestamp": the_df.iloc[idx, 0],
                    "arbitration_id": hex(idDec),
                    "raw_data": payload_str,
                    "label": label,
                    "decoded": translated_message,
                }

                results.append(row)
        except Exception:
            non_translated_count += 1

            if output_csv:
                row = {
                    "timestamp": the_df.iloc[idx, 0],
                    "arbitration_id": hex(idDec),
                    "raw_data": payload_str,
                    "label": label,
                    "decoded": "UNDECODABLE",
                }

                results.append(row)

    # summary statistics
    print(f"Total size: {total_count}")
    print(f"Translated count: {translated_count}")
    print(f"Not translated count: {non_translated_count}")
    print(
        f"% of total size that was translated: {translated_count/total_count if total_count else 0}"
    )

    # write out results if requested
    if output_csv and results:
        import pandas as pd

        results_df = pd.DataFrame(results)
        results_df.to_csv(output_csv, index=False)

        print(f"Results written to {output_csv}")

        return results_df

In [21]:
import pandas as pd

file_list = [
    "CAN-CarHacking/DoS_dataset.csv",
    "CAN-CarHacking/RPM_dataset.csv",
    "CAN-CarHacking/gear_dataset.csv",
    "CAN-CarHacking/Fuzzy_dataset.csv",
]

payload_cols = ['[P','a','Y','L','O','A','D','data]']

for fname in file_list:

    df = openCSV(fname)

    # Standardize column names
    df.columns = [
        "Timestamp","CAN ID","Data size (bytes)",
        "[P","a","Y","L","O","A","D","data]","Label"
    ]

    # Convert payload columns to string to avoid int+str errors
    df[payload_cols] = df[payload_cols].astype(str)

    # Split by payload size
    two_byte_df   = df[df["Data size (bytes)"] == 2].copy()
    five_byte_df   = df[df["Data size (bytes)"] == 5].copy()
    six_byte_df   = df[df["Data size (bytes)"] == 6].copy()
    eight_byte_df = df[df["Data size (bytes)"] == 8].copy()

    # Combine payload for 8 byte frames
    eight_byte_df["data"] = eight_byte_df[payload_cols].agg(''.join, axis=1)

    # Combine payload for 2 byte frames
    two_byte_df["data"] = (
        two_byte_df['[P'] +
        two_byte_df['a']
    )

    # Combine payload for 5 byte frames
    five_byte_df["data"] = (
        five_byte_df['[P'] +
        five_byte_df['a'] +
        five_byte_df['Y'] +
        five_byte_df['L'] +
        five_byte_df['O']
    )

    # Combine payload for 6 byte frames
    six_byte_df["data"] = (
        six_byte_df['[P'] +
        six_byte_df['a'] +
        six_byte_df['Y'] +
        six_byte_df['L'] +
        six_byte_df['O'] +
        six_byte_df['A']
    )

    # Keep only necessary columns
    two_byte_df   = two_byte_df[["Timestamp","CAN ID","Data size (bytes)","data","Label"]]
    five_byte_df  = five_byte_df[["Timestamp","CAN ID","Data size (bytes)","data","Label"]]
    six_byte_df   = six_byte_df[["Timestamp","CAN ID","Data size (bytes)","data","Label"]]
    eight_byte_df = eight_byte_df[["Timestamp","CAN ID","Data size (bytes)","data","Label"]]


    # Merge frames
    data_formatted = pd.concat([two_byte_df, five_byte_df, six_byte_df, eight_byte_df])

    # Sort by timestamp
    data_formatted = data_formatted.sort_values("Timestamp")

    # Dataset size
    size = len(data_formatted)

    # Output file name
    output_name = fname.replace(".csv","_decoded.csv")

    translate_with_csv(
        data_formatted,
        "hyundai_kia_generic.dbc",
        size,
        output_csv=output_name
    )

    print(f"Finished processing {fname}")

Total size: 3665770
Translated count: 1288324
Not translated count: 2377446
% of total size that was translated: 0.35144703568418095
Results written to CAN-CarHacking/DoS_dataset_decoded.csv
Finished processing CAN-CarHacking/DoS_dataset.csv
Total size: 4621701
Translated count: 2314792
Not translated count: 2306909
% of total size that was translated: 0.5008528245336511
Results written to CAN-CarHacking/RPM_dataset_decoded.csv
Finished processing CAN-CarHacking/RPM_dataset.csv
Total size: 4443141
Translated count: 1611701
Not translated count: 2831440
% of total size that was translated: 0.36273910731169684
Results written to CAN-CarHacking/gear_dataset_decoded.csv
Finished processing CAN-CarHacking/gear_dataset.csv
Total size: 3838859
Translated count: 1457367
Not translated count: 2381492
% of total size that was translated: 0.37963545939040744
Results written to CAN-CarHacking/Fuzzy_dataset_decoded.csv
Finished processing CAN-CarHacking/Fuzzy_dataset.csv
